In [1]:
# %%
# =============================================================================
# Paper C · 02_baseline_basic
# Experiment condition ① : basic info ONLY (minimal baseline / floor)
#
# Goal: establish the performance FLOOR using only the 3 basic company attributes
#       (register_capital, paid_capital, setup_time_months) — no lawsuit signal,
#       no graph, no LLM. Everything downstream (03 GNN, 05 +enrich) is measured
#       as uplift over this floor.
#
# Classifiers: Logistic Regression + a small MLP (per agreed plan).
# Evaluation : mask A (strict, no-leakage) as PRIMARY; mask B (as-shipped) for
#              reference only. Metrics: AUC, F1, KS (label balance is ~even, so
#              AUC/F1 are appropriate; no resampling needed).
#
# Inputs : data/processed/smesd_hetero.pt   (from 01_build_graph)
# Outputs: artifacts/tables/02_baseline_basic.csv
# =============================================================================

from pathlib import Path

import numpy as np
import pandas as pd
import torch

# %%
# --- Paths -------------------------------------------------------------------
ROOT = Path.cwd()
if ROOT.name in {"notebooks", "paperC"}:
    ROOT = ROOT.parents[0] if ROOT.name == "notebooks" else ROOT.parents[1]
PROCESSED = ROOT / "data" / "processed"
TABLES = ROOT / "artifacts" / "tables"
TABLES.mkdir(parents=True, exist_ok=True)

data = torch.load(PROCESSED / "smesd_hetero.pt", weights_only=False)
comp = data["company"]
print(f"loaded company store: x={tuple(comp.x.shape)} y={tuple(comp.y.shape)}")


# %%
# --- 1. Pull basic-only features + labels + masks ----------------------------
# company.x layout (from 01): [0:3] basic | [3:] lawsuit-aggregate.
# Condition ① uses ONLY the first 3 columns.
X_all = comp.x.numpy()
y_all = comp.y.numpy()
X_basic = X_all[:, :3]           # register_capital, paid_capital, setup_time_m
BASIC_COLS = ["register_capital", "paid_capital", "setup_time_m"]

masks = {
    "A": {  # strict, no-leakage (PRIMARY)
        "train": comp.train_mask.numpy(),
        "val":   comp.val_mask.numpy(),
        "test":  comp.test_mask.numpy(),
    },
    "B": {  # as-shipped (reference)
        "train": comp.train_mask_repro.numpy(),
        "val":   comp.val_mask_repro.numpy(),
        "test":  comp.test_mask_repro.numpy(),
    },
}
for tag, m in masks.items():
    print(f"mask {tag}: train={m['train'].sum():,} "
          f"val={m['val'].sum():,} test={m['test'].sum():,}")


# %%
# --- 2. Inspect distributions → decide on log transform ----------------------
# Capital columns are typically heavy-tailed (thousands–millions); setup_time is
# a month count. We let the data decide: if skew is large, apply log1p to the
# capital columns. We only ever FIT scaling on the training rows (no leakage).
def describe_cols(X, cols):
    df = pd.DataFrame(X, columns=cols)
    return df.describe(percentiles=[.5, .9, .99]).T[
        ["mean", "std", "50%", "90%", "99%", "max"]]

train_rows = masks["A"]["train"]
print("basic feature distribution (train rows, mask A):")
print(describe_cols(X_basic[train_rows], BASIC_COLS).to_string())

# Skewness on training rows drives the transform decision.
from scipy.stats import skew
sk = {c: float(skew(X_basic[train_rows, i])) for i, c in enumerate(BASIC_COLS)}
print("\nskewness (train):", {k: round(v, 2) for k, v in sk.items()})

LOG_COLS = [i for i, c in enumerate(BASIC_COLS)
            if c in ("register_capital", "paid_capital") and abs(sk[c]) > 2]
print(f"applying log1p to columns: {[BASIC_COLS[i] for i in LOG_COLS]}")


# %%
# --- 3. Preprocessing: log1p (heavy-tailed capitals) + standardize -----------
# Fit transforms on TRAIN ONLY, apply to val/test — strict to avoid leakage.
from sklearn.preprocessing import StandardScaler


def preprocess(X, train_mask, log_cols):
    Xt = X.copy().astype(np.float64)
    for i in log_cols:
        # capitals can be 0; log1p keeps 0 -> 0 and handles the tail.
        Xt[:, i] = np.log1p(np.clip(Xt[:, i], a_min=0, a_max=None))
    scaler = StandardScaler().fit(Xt[train_mask])
    return scaler.transform(Xt), scaler


# %%
# --- 4. Metric helpers -------------------------------------------------------
# AUC (ranking), F1 (thresholded at 0.5), KS (max TPR-FPR separation).
from sklearn.metrics import roc_auc_score, f1_score, roc_curve


def ks_stat(y_true, y_score):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    return float(np.max(tpr - fpr))


def evaluate(y_true, y_score, thr=0.5):
    y_pred = (y_score >= thr).astype(int)
    return {
        "AUC": roc_auc_score(y_true, y_score),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "KS": ks_stat(y_true, y_score),
    }


# %%
# --- 5. Models: Logistic Regression + small MLP ------------------------------
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

SEED = 42


def make_models():
    return {
        "LogReg": LogisticRegression(max_iter=1000, class_weight="balanced",
                                     random_state=SEED),
        "MLP": MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=500,
                             early_stopping=True, random_state=SEED),
    }


# %%
# --- 6. Train / evaluate across masks × models -------------------------------
# For each mask set (A primary, B reference), fit on train, report val + test.
rows = []
for mask_tag, m in masks.items():
    Xp, _ = preprocess(X_basic, m["train"], LOG_COLS)
    tr, va, te = m["train"], m["val"], m["test"]

    for name, model in make_models().items():
        model.fit(Xp[tr], y_all[tr])
        # predicted probability of the positive (bankrupt) class
        val_p = model.predict_proba(Xp[va])[:, 1]
        test_p = model.predict_proba(Xp[te])[:, 1]

        for split_name, yt, yp in [("val", y_all[va], val_p),
                                    ("test", y_all[te], test_p)]:
            met = evaluate(yt, yp)
            rows.append({"mask": mask_tag, "model": name, "split": split_name,
                         **{k: round(v, 4) for k, v in met.items()}})

results = pd.DataFrame(rows)
print(results.to_string(index=False))


# %%
# --- 7. Persist the baseline table -------------------------------------------
out = TABLES / "02_baseline_basic.csv"
results.to_csv(out, index=False)
print(f"→ saved {out.relative_to(ROOT)}")

# Headline: primary = mask A, test split.
primary = results[(results["mask"] == "A") & (results["split"] == "test")]
print("\nPRIMARY floor (mask A, test):")
print(primary[["model", "AUC", "F1", "KS"]].to_string(index=False))


# %%
# --- 8. Notes for downstream notebooks ---------------------------------------
print("""
Baseline floor established (condition ① basic-info only)
========================================================
  This is the FLOOR. Report uplift over these test-split numbers for:
    03_gnn_reproduce  (graph structure)
    05_gnn_plus_enrich (graph + LLM-enriched lawsuit codes)
  Primary comparison uses mask A (strict). Mask B is reference only — expect
  slightly higher B numbers because the 51 duplicated firms leak across splits.

  Reminder: label balance is ~even (57/49/65% positive), so AUC/F1/KS are the
  right metrics and no resampling was applied.
""")
print("02_baseline_basic complete.")

loaded company store: x=(3976, 23) y=(3976,)
mask A: train=2,765 val=720 test=491
mask B: train=2,816 val=721 test=491
basic feature distribution (train rows, mask A):
                         mean           std    50%     90%           99%         max
register_capital  4580.465332  72537.750000  500.0  5000.0  43722.739219  3565913.25
paid_capital      4322.907715  72376.843750  500.0  5000.0  31569.841250  3565913.25
setup_time_m       153.256775     73.488533  143.0   249.0    354.000000      504.00

skewness (train): {'register_capital': 44.0, 'paid_capital': 44.29, 'setup_time_m': 0.82}
applying log1p to columns: ['register_capital', 'paid_capital']
mask  model split    AUC     F1     KS
   A LogReg   val 0.6386 0.6205 0.2312
   A LogReg  test 0.6235 0.7157 0.2388
   A    MLP   val 0.6097 0.6768 0.2139
   A    MLP  test 0.5864 0.7770 0.1932
   B LogReg   val 0.6369 0.6215 0.2358
   B LogReg  test 0.6207 0.7157 0.2439
   B    MLP   val 0.6747 0.6741 0.2669
   B    MLP  test 0.6788 

In [2]:
# %%
# --- DIAGNOSTIC: why person_emb has 2405 rows but only 2402 persons in edges --
import pickle
from pathlib import Path
import numpy as np
import torch

ROOT = Path.cwd()
if ROOT.name in {"notebooks", "paperC"}:
    ROOT = ROOT.parents[0] if ROOT.name == "notebooks" else ROOT.parents[1]
DATA_DIR = ROOT / "data" / "external" / "ComRisk" / "data"
PROCESSED = ROOT / "data" / "processed"

def load(n):
    with open(DATA_DIR / n, "rb") as f:
        return pickle.load(f)

meta_emb = load("meta_emb.pkl")
company_emb = np.asarray(meta_emb[0])
person_emb = np.asarray(meta_emb[1])
N_COMPANY = company_emb.shape[0]          # 3976
N_PERSON_EMB = person_emb.shape[0]        # 2405

idmaps = torch.load(PROCESSED / "smesd_idmaps.pt", weights_only=False)
comp_global = set(idmaps["comp_global"].tolist())
person_global = set(np.asarray(idmaps["person_global"]).tolist())

# Collect ALL global node ids that appear in any edge, across splits.
edge_ids = set()
for nm in ["train_data.pkl", "validate_data.pkl", "test_data.pkl"]:
    obj = load(nm)
    hg = obj[2]
    ei = np.asarray(hg[0] if isinstance(hg, (list, tuple)) else hg["edge_index"])
    if ei.shape[0] != 2:
        ei = ei.T
    edge_ids.update(ei.ravel().tolist())

# The universe of possible person ids if convention is "companies first".
# Hypothesis H1: global ids 0..3975 are companies, 3976..6380 are persons.
GLOBAL_MAX = 6380
h1_person_ids = set(range(N_COMPANY, GLOBAL_MAX + 1))   # 3976..6380

print(f"company_emb rows : {N_COMPANY}")
print(f"person_emb rows  : {N_PERSON_EMB}")
print(f"persons in edges : {len(person_global)}")
print(f"all edge node ids: {len(edge_ids)}  range [{min(edge_ids)}, {max(edge_ids)}]")

print("\n--- H1: are person ids exactly 3976..6380 ? ---")
print(f"|3976..6380| = {len(h1_person_ids)}  (expect 2405 if H1 true: "
      f"{len(h1_person_ids) == N_PERSON_EMB})")
# which of the hypothesized person-id slots actually appear in edges?
appear = h1_person_ids & edge_ids
missing = sorted(h1_person_ids - edge_ids)
print(f"of those, appear in edges: {len(appear)}  missing: {len(missing)}")
print(f"missing person-id slots (emb exists, no edge): {missing[:10]}"
      f"{'...' if len(missing) > 10 else ''}")

# Cross-check: do our detected person_global ids fall in 3976..6380?
outside = [g for g in person_global if not (N_COMPANY <= g <= GLOBAL_MAX)]
print(f"\ndetected persons OUTSIDE 3976..6380: {len(outside)} {outside[:10]}")

# Are the 3 extra just isolated emb rows, or an off-by-N shift?
print(f"\nverdict hint: if missing=={N_PERSON_EMB - len(person_global)} (=3) and "
      f"no persons fall outside the band, the 3 extras are ISOLATED nodes "
      f"(emb present, no edges) → safe to map by global-id offset.")

company_emb rows : 3976
person_emb rows  : 2405
persons in edges : 2402
all edge node ids: 6378  range [0, 6380]

--- H1: are person ids exactly 3976..6380 ? ---
|3976..6380| = 2405  (expect 2405 if H1 true: True)
of those, appear in edges: 2402  missing: 3
missing person-id slots (emb exists, no edge): [3979, 4144, 4683]

detected persons OUTSIDE 3976..6380: 0 []

verdict hint: if missing==3 (=3) and no persons fall outside the band, the 3 extras are ISOLATED nodes (emb present, no edges) → safe to map by global-id offset.
